# SDXL Convert Phase 2-5 (TPU v5e-8, 377GB RAM)

Consumes Phase 1 kernel's output (`data_sdxl.pkl` + `images_sdxl/`) from `/kaggle/input/sdxl-convert-phase1-gpu/phase1_out/`.

Runs:
- Phase 2: `gen_quant_data_sdxl.py`
- Phase 3: `export_onnx_sdxl.py`
- Phase 4+5: `convert_all_sdxl.sh` (qairt converter + quantizer)

TPU sits idle — we're using this runtime for its 330GB system RAM + 96 CPU cores.

In [ ]:
CIVITAI_VERSION_ID = '__CIVITAI_VERSION_ID__'
MODEL_NAME         = '__MODEL_NAME__'
REALISTIC          = '__REALISTIC__' == 'true'
MIN_SOC            = '__MIN_SOC__'
print(f'civitai={CIVITAI_VERSION_ID} name={MODEL_NAME} realistic={REALISTIC} min_soc={MIN_SOC}')

In [ ]:
!free -h
!nproc
!df -h /kaggle/working /tmp
!cat /etc/os-release | head -5
!ldd --version | head -1
!grep -m1 'model name' /proc/cpuinfo
!grep -m1 'cpu MHz' /proc/cpuinfo
!lscpu | grep -E 'Model name|Socket|Thread|Core|Vendor|Flags' | head -10

In [ ]:
# Locate Phase 1 output
import glob, os
candidates = glob.glob('/kaggle/input/**/data_sdxl.pkl', recursive=True)
assert candidates, 'Phase 1 data_sdxl.pkl not found in /kaggle/input/ — did Phase 1 kernel complete?'
PHASE1_DIR = os.path.dirname(candidates[0])
print(f'Phase 1 dir: {PHASE1_DIR}')
!ls -lh $PHASE1_DIR

In [ ]:
import os
os.environ['MIN_SOC'] = MIN_SOC
os.environ['MODEL_NAME'] = MODEL_NAME

In [ ]:
# Install tools + convertsdxl + shell helper
import os, subprocess

def _run(cmd, log=None, check=True):
    """bash with errexit + pipefail; pipe to tee if log. Raises on non-zero."""
    if log:
        os.makedirs(os.path.dirname(log), exist_ok=True)
        wrapped = f'set -eo pipefail; {cmd} 2>&1 | tee {log}'
    else:
        wrapped = f'set -eo pipefail; {cmd}'
    rc = subprocess.call(['bash', '-c', wrapped])
    if check and rc != 0:
        raise RuntimeError(f'shell failed (rc={rc}): {cmd[:200]}  log={log}')
    return rc

# Ensure apt cache is fresh before any installs
_run('apt-get update -y > /dev/null')
_run('apt-get install -y unzip zip curl time > /dev/null')
# libc++/libc++abi/libunwind needed by QNN SDK libPyIrGraph.so / libQnnHtp.so — Kaggle Debian 13 trixie doesn't ship them
# Specific LLVM runtime packages — generic libc++-dev/libunwind-dev do NOT provide libunwind.so.1 (GNU libunwind ships .so.8)
_run('apt-get install -y libc++1-19 libc++abi1-19 libunwind-19 > /dev/null')
_run('ldconfig')  # refresh ld cache so newly installed libs are found by ldd / dlopen
_run('curl -LsSf https://astral.sh/uv/install.sh | sh > /dev/null 2>&1')
os.environ['PATH'] = f"/root/.local/bin:{os.environ['PATH']}"

_run('rm -rf /tmp/convertsdxl.zip /tmp/convertsdxl_unzip')
_run("curl -sL --fail --retry 5 --retry-delay 5 -o /tmp/convertsdxl.zip 'https://chino.icu/local-dream/convertsdxl.zip'")
_run('mkdir -p /tmp/convertsdxl_unzip')
_run('unzip -q /tmp/convertsdxl.zip -d /tmp/convertsdxl_unzip')

root = subprocess.check_output(
    "find /tmp/convertsdxl_unzip -maxdepth 3 -name 'export_sdxl.sh' -printf '%h\\n' | head -1",
    shell=True, text=True).strip()
assert root, 'export_sdxl.sh not found in unzipped convertsdxl'
NPUCONVERT_DIR = os.path.abspath(root)
os.environ['NPUCONVERT_DIR'] = NPUCONVERT_DIR
print(f'NPUCONVERT_DIR: {NPUCONVERT_DIR}')

In [ ]:
# QNN SDK
import os, subprocess
_run('mkdir -p /tmp/qnn_sdk')
_run(
    "curl -sL --fail --retry 5 --retry-delay 5 -A 'Mozilla/5.0' "
    "-o /tmp/qnn_sdk/qnn.zip "
    "'https://apigwx-aws.qualcomm.com/qsc/public/v1/api/download/software/qualcomm_neural_processing_sdk/v2.28.0.241029.zip'"
)
_run('cd /tmp/qnn_sdk && unzip -q qnn.zip')
bin_file = subprocess.check_output(
    'find /tmp/qnn_sdk -type f -name envsetup.sh -print -quit',
    shell=True, text=True).strip()
assert bin_file, 'envsetup.sh not found in QNN SDK extract'
QNN_SDK_BIN = os.path.dirname(bin_file)
QNN_SDK_ROOT_DIR = os.path.dirname(QNN_SDK_BIN)
os.environ['QNN_SDK_BIN'] = QNN_SDK_BIN
os.environ['QNN_SDK_ROOT_DIR'] = QNN_SDK_ROOT_DIR
print(f'QNN_SDK_ROOT: {QNN_SDK_ROOT_DIR}')

In [ ]:
# Resolve model.safetensors from Phase 1 kernel_sources mount (read-only, no re-download)
import os, glob
candidates = glob.glob('/kaggle/input/**/model.safetensors', recursive=True)
assert candidates, 'model.safetensors not found in /kaggle/input/ — Phase 1 kernel_sources not mounted?'
os.environ['MODEL_PATH'] = candidates[0]
size = os.path.getsize(os.environ['MODEL_PATH'])
assert size > int(1e9), f'model.safetensors truncated: {size} bytes'
print(f'MODEL_PATH -> {os.environ["MODEL_PATH"]} ({size/1e9:.2f} GB)')

In [ ]:
# Python env
import os
os.chdir(os.environ['NPUCONVERT_DIR'])
_run('uv venv -p 3.10.17 --clear')
_run('. .venv/bin/activate && uv sync')

In [ ]:
# Copy Phase 1 output into convert dir
import os, shutil, glob
pkl_list = glob.glob('/kaggle/input/**/data_sdxl.pkl', recursive=True)
assert pkl_list, 'data_sdxl.pkl not found in /kaggle/input/ — Phase 1 mount missing'
pkl = pkl_list[0]
p1_dir = os.path.dirname(pkl)
shutil.copy(pkl, f'{os.environ["NPUCONVERT_DIR"]}/data_sdxl.pkl')
src_img = f'{p1_dir}/images_sdxl'
assert os.path.isdir(src_img), f'{src_img} missing — Phase 1 did not produce calibration images'
dst_img = f'{os.environ["NPUCONVERT_DIR"]}/images_sdxl'
if os.path.exists(dst_img):
    shutil.rmtree(dst_img)
shutil.copytree(src_img, dst_img)
assert len(os.listdir(dst_img)) > 0, f'{dst_img} empty after copy'
_run(f'ls -lh {os.environ["NPUCONVERT_DIR"]}/data_sdxl.pkl')
_run(f'ls {dst_img} | head -5')

In [ ]:
# Start memory watcher + TPU heartbeat (prevents Kaggle's 2hr-idle TPU auto-cancel).
# Match Kaggle's official v5e-8 notebook format (tensorflow-v5e-8-oct-2025.ipynb): uninstall JAX,
# install tensorflow-tpu==2.18.0 with matching libtpu from libtpu-tf-releases. Then run REAL TPU
# compute (8192x8192 matmul × 5 iters) every 10 min, not a trivial dot — Kaggle appears to detect
# actual TPU hardware utilization, not just library calls.
import subprocess, os, time
os.makedirs('/kaggle/working/logs', exist_ok=True)

# --- mem watcher (background) ---
watcher_sh = '''#!/bin/bash
echo "epoch,datetime,MemAvail_MB,TopRSS_MB,TopProc"
while true; do
  epoch=$(date +%s); dt=$(date -Iseconds)
  mem=$(awk '/MemAvailable:/{print int($2/1024)}' /proc/meminfo)
  line=$(ps -eo rss,comm --sort=-rss --no-headers 2>/dev/null | head -1)
  rss=$(echo "$line" | awk '{print int($1/1024)}')
  proc=$(echo "$line" | awk '{print $2}')
  echo "$epoch,$dt,$mem,$rss,$proc"
  sleep 10
done
'''
open('/tmp/mem_watch.sh','w').write(watcher_sh)
os.chmod('/tmp/mem_watch.sh', 0o755)
p = subprocess.Popen(['/tmp/mem_watch.sh'], stdout=open('/kaggle/working/logs/mem-trace.csv','w'), stderr=subprocess.DEVNULL)
open('/tmp/watcher.pid','w').write(str(p.pid))
print(f'mem-watcher pid={p.pid}')

# --- Install TF-TPU per Kaggle's official v5e-8 example ---
# JAX default image may not properly drive v5e-8; Kaggle's official example uninstalls JAX.
_run('export PATH="${HOME}/.local/bin:${PATH}" && uv pip uninstall --system jax 2>&1 | tail -3', check=False)
_run('export PATH="${HOME}/.local/bin:${PATH}" && uv pip install --system --quiet '
     'tensorflow-tpu==2.18.0 '
     '--find-links https://storage.googleapis.com/libtpu-tf-releases/index.html')

# Probe: TPUStrategy sees all replicas
probe = subprocess.run(['/usr/local/bin/python3', '-c', '''
import tensorflow as tf
tpu = tf.distribute.cluster_resolver.TPUClusterResolver(tpu="local")
tf.config.experimental_connect_to_cluster(tpu)
tf.tpu.experimental.initialize_tpu_system(tpu)
strategy = tf.distribute.TPUStrategy(tpu)
print(f"REPLICAS={strategy.num_replicas_in_sync}")
'''], capture_output=True, text=True)
print(f'TF-TPU probe: rc={probe.returncode}')
print(f'  stdout[-400:]={probe.stdout.strip()[-400:]}')
print(f'  stderr[-400:]={probe.stderr.strip()[-400:]}')
replicas = 0
for ln in reversed((probe.stdout or '').splitlines()):
    if ln.strip().startswith('REPLICAS='):
        try: replicas = int(ln.strip().split('=',1)[1])
        except ValueError: replicas = 0
        break
if probe.returncode != 0 or replicas == 0:
    raise RuntimeError(
        f'TF-TPU unavailable (rc={probe.returncode} replicas={replicas}). '
        f'Heartbeat cannot start; Kaggle will kill the VM at 2h TPU idle. '
        f'stderr: {probe.stderr[:600]}'
    )
print(f'TF-TPU sees {replicas} replicas')

# --- TPU heartbeat subprocess (real compute every 10 min) ---
heartbeat_sh = '''#!/bin/bash
while true; do
  ts=$(date -Iseconds)
  out=$(/usr/local/bin/python3 <<'PY' 2>&1
import tensorflow as tf, time
tpu = tf.distribute.cluster_resolver.TPUClusterResolver(tpu="local")
tf.config.experimental_connect_to_cluster(tpu)
tf.tpu.experimental.initialize_tpu_system(tpu)
strategy = tf.distribute.TPUStrategy(tpu)

@tf.function
def step():
    x = tf.random.normal((8192, 8192))
    y = tf.random.normal((8192, 8192))
    return tf.reduce_sum(tf.matmul(x, y))

t0 = time.time()
for _ in range(5):
    per_replica = strategy.run(step)
    total = strategy.reduce(tf.distribute.ReduceOp.SUM, per_replica, axis=None)
    _ = float(total)
elapsed = time.time() - t0
print(f"replicas={strategy.num_replicas_in_sync} matmul_5x_elapsed={elapsed:.2f}s")
PY
)
  echo "[$ts] heartbeat: $out"
  sleep 600
done
'''
open('/tmp/tpu_heartbeat.sh','w').write(heartbeat_sh)
os.chmod('/tmp/tpu_heartbeat.sh', 0o755)
hp = subprocess.Popen(['/tmp/tpu_heartbeat.sh'],
    stdout=open('/kaggle/working/logs/tpu_heartbeat.log','w'),
    stderr=subprocess.STDOUT)
open('/tmp/tpu_heartbeat.pid','w').write(str(hp.pid))
print(f'tpu-heartbeat pid={hp.pid}')
# wait up to 60s for first cycle (TF init + matmul can take ~30-60s)
time.sleep(60)
_run('head -20 /kaggle/working/logs/tpu_heartbeat.log', check=False)
# If first heartbeat never ran or errored out, catch it now rather than waiting 10 min
hb = open('/kaggle/working/logs/tpu_heartbeat.log').read()
if 'replicas=' not in hb or 'matmul_5x_elapsed=' not in hb:
    raise RuntimeError(f'TPU heartbeat first cycle failed. Log:\n{hb[-1500:]}')

In [ ]:
# Phase 2
import time, os
t0 = time.time()
os.chdir(os.environ['NPUCONVERT_DIR'])
_run(
    '. .venv/bin/activate && /usr/bin/time -v python gen_quant_data_sdxl.py',
    log='/kaggle/working/logs/phase2.log'
)
print(f'Phase 2 elapsed: {int(time.time()-t0)}s')
_run('free -h')

In [ ]:
# Phase 3
import time, os
t0 = time.time()
os.chdir(os.environ['NPUCONVERT_DIR'])
_run(
    '. .venv/bin/activate && /usr/bin/time -v python export_onnx_sdxl.py --model_path "$MODEL_PATH"',
    log='/kaggle/working/logs/phase3.log'
)
print(f'Phase 3 elapsed: {int(time.time()-t0)}s')
_run('free -h')
_run(r'find . -name "*.onnx" -exec ls -lh {} \;')
_run(r'find . -name "weights.pb" -exec ls -lh {} \;')

In [ ]:
# Backup Phase 3 ONNX output to /kaggle/working/phase3_backup
import os, shutil
bkp = '/kaggle/working/phase3_backup'
if os.path.exists(bkp): shutil.rmtree(bkp)
os.makedirs(bkp)
expected = ['clip_sdxl','clip2_sdxl','unet_sdxl','vae_decoder_sdxl','vae_encoder_sdxl']
missing = [d for d in expected if not os.path.exists(f'{os.environ["NPUCONVERT_DIR"]}/{d}')]
assert not missing, f'Phase 3 output dirs missing (partial export?): {missing}'
for d_ in expected:
    shutil.copytree(f'{os.environ["NPUCONVERT_DIR"]}/{d_}', f'{bkp}/{d_}')
_run(f'ls {bkp}')
_run(f'du -sh {bkp}/*')
_run(rf'find {os.environ["NPUCONVERT_DIR"]} -maxdepth 2 -name "input_list_*.txt" -exec cp {{}} {bkp}/ \;')
_run(f'ls {bkp}/*.txt', check=False)  # .txt copy is cosmetic

In [ ]:
# Patch convert scripts: replace hardcoded QNN_SDK_ROOT, make all config paths ABSOLUTE to NPUCONVERT_DIR.
# Reason: qnn-context-binary-generator's --config_file ./x.json is cwd-relative; if cwd drifts between
# qairt-quantizer (22min) and qnn-context-binary-generator we fail with "Cannot open file".
# Absolute paths eliminate the entire class of problems.
import os, re, glob, json as _json

NPU = os.environ['NPUCONVERT_DIR']
QNN = os.environ['QNN_SDK_ROOT_DIR']
SOC = os.environ['MIN_SOC']  # e.g. '8gen3' or '8gen4' (upstream default moved 8gen4 -> 8gen3 on 2026-04-20)

# 1. convert_all_sdxl.sh: replace hardcoded QNN_SDK_ROOT + prepend cd to NPUCONVERT_DIR
main = f'{NPU}/scripts/convert_all_sdxl.sh'
orig = open(main).read()
pat = re.compile(r'QNN_SDK_ROOT=/data/qairt/[0-9.]+')
assert pat.search(orig), f'QNN_SDK_ROOT pattern not found in {main}'
patched = pat.sub(f'QNN_SDK_ROOT={QNN}', orig)
patched = patched.replace('set -e\n', f'set -e\ncd "{NPU}"\n', 1)
assert patched != orig
open(main, 'w').write(patched)

# 2. Each sub-script: prepend defensive cd
for sub in glob.glob(f'{NPU}/scripts/convert_*.sh'):
    if sub.endswith('convert_all_sdxl.sh'):
        continue
    s = open(sub).read()
    s2 = s.replace('set -e\n', f'set -e\ncd "{NPU}"\n', 1)
    open(sub, 'w').write(s2)
    print(f'patched {os.path.basename(sub)}: prepended cd')

# 3. Rewrite htp_backend_<SOC>.json so config_file_path is absolute
hbp = f'{NPU}/htp_backend_{SOC}.json'
assert os.path.exists(hbp), f'{hbp} missing — upstream convertsdxl.zip may not ship this SOC config'
d = _json.load(open(hbp))
d['backend_extensions']['config_file_path'] = f'{NPU}/htp_config_{SOC}.json'
_json.dump(d, open(hbp, 'w'), indent=2)
print(f'htp_backend_{SOC}.json now:', open(hbp).read())

# 4. Sanity grep
_run(f'grep -nE "QNN_SDK_ROOT|^cd |source envsetup" {main}')

In [ ]:
# Phase 4-5 — QNN convert + quantize
import time, os, subprocess
t0 = time.time()
os.chdir(os.environ['NPUCONVERT_DIR'])
SOC = os.environ['MIN_SOC']

# qairt-converter -> libPyIrGraph.so -> libpython3.10.so.1.0 (NEEDED).
_pylib = subprocess.check_output(
    ['.venv/bin/python', '-c', 'import sys,os; print(os.path.join(sys.base_prefix, "lib"))'],
    text=True).strip()
assert os.path.exists(os.path.join(_pylib, 'libpython3.10.so.1.0')), \
    f'libpython3.10.so.1.0 not in {_pylib}'
os.environ['LD_LIBRARY_PATH'] = f"{_pylib}:{os.environ.get('LD_LIBRARY_PATH','')}"
print(f'libpython dir prepended to LD_LIBRARY_PATH: {_pylib}')

# Pre-flight: every critical QNN .so must have ldd resolve all deps.
_qnn = os.environ['QNN_SDK_ROOT_DIR']
_qnn_lib = f'{_qnn}/lib/x86_64-linux-clang'
_check_env = os.environ.copy()
_check_env['LD_LIBRARY_PATH'] = f'{_qnn_lib}:{_check_env["LD_LIBRARY_PATH"]}'
_critical_libs = [
    f'{_qnn}/lib/python/qti/aisw/converters/common/linux-x86_64/libPyIrGraph.so',
    f'{_qnn}/lib/python/qti/aisw/converters/common/linux-x86_64/libPyIrQuantizer.so',
    f'{_qnn}/lib/x86_64-linux-clang/libQnnHtp.so',
    f'{_qnn}/lib/x86_64-linux-clang/libQnnModelDlc.so',
]
for _lib in _critical_libs:
    assert os.path.exists(_lib), f'{_lib} not found in SDK extract'
    _ldd = subprocess.check_output(['ldd', _lib], env=_check_env, text=True)
    _missing = [l.strip() for l in _ldd.splitlines() if 'not found' in l]
    if _missing:
        print(f'FULL ldd output for {_lib}:\n{_ldd}')
        raise AssertionError(f'{_lib} unresolved deps:\n' + '\n'.join(_missing))
    print(f'OK  {os.path.basename(_lib)}')

# Pre-flight: config + input files must exist right before kick-off.
_npu = os.environ['NPUCONVERT_DIR']
_required = [
    f'{_npu}/htp_backend_{SOC}.json',
    f'{_npu}/htp_config_{SOC}.json',
    f'{_npu}/tokenizer.json',
    f'{_npu}/MNNConvert',
    f'{_npu}/scripts/convert_all_sdxl.sh',
    f'{_npu}/data_sdxl.pkl',
    f'{_npu}/vae_encoder_sdxl/model.onnx',
    f'{_npu}/vae_decoder_sdxl/model.onnx',
    f'{_npu}/unet_sdxl/model.onnx',
    f'{_npu}/clip_sdxl/model.onnx',
    f'{_npu}/clip2_sdxl/model.onnx',
    f'{_npu}/input_list_unet_sdxl.txt',
    f'{_npu}/input_list_vae_decoder_sdxl.txt',
    f'{_npu}/input_list_vae_encoder_sdxl.txt',
]
_missing_files = [p for p in _required if not os.path.exists(p)]
assert not _missing_files, f'Required files missing at Phase 4-5 kick-off:\n' + '\n'.join(_missing_files)
print(f'All {len(_required)} required files present in {_npu}')

# Run with inline tee + awk filter (drop "thread affinity for cpuset" noise from display; log keeps full)
os.makedirs('/kaggle/working/logs', exist_ok=True)
_run(
    f'cd "{_npu}" && . .venv/bin/activate && '
    f'(/usr/bin/time -v bash scripts/convert_all_sdxl.sh --min_soc "$MIN_SOC") 2>&1 '
    f'| tee /kaggle/working/logs/phase45.log '
    f'| awk \'!/Failed to set thread affinity for cpuset/\'',
    log=None
)
print(f'Phase 4-5 elapsed: {int(time.time()-t0)}s')
_run('free -h')
for p in [f'{_npu}/htp_backend_{SOC}.json', f'{_npu}/htp_config_{SOC}.json']:
    print(f'post-run exists({p}): {os.path.exists(p)}')
_run(r'echo -n "suppressed thread-affinity lines: "; grep -c "thread affinity for cpuset" /kaggle/working/logs/phase45.log || echo 0')

In [ ]:
# Package output zip
import os, shutil
os.chdir(os.environ['NPUCONVERT_DIR'])
out_dir = f'output/qnn_models_sdxl_{os.environ["MIN_SOC"]}'
assert os.path.isdir(out_dir), f'{out_dir} missing — Phase 4-5 produced no output'

# Free /kaggle/working space: Phase 4-5 succeeded (we're past cell 15's raise), so phase3_backup is no longer needed.
# /kaggle/working has a 20GB cap; keeping 12.8GB backup + zip would exceed it.
bkp = '/kaggle/working/phase3_backup'
if os.path.exists(bkp):
    shutil.rmtree(bkp)
    print(f'reclaimed phase3_backup space')
_run('df -h /kaggle/working')

_run(f'touch {out_dir}/SDXL')
zip_name = f'{os.environ["MODEL_NAME"]}_qnn2.28_{os.environ["MIN_SOC"]}.zip'
_run(f'zip -r /kaggle/working/{zip_name} {out_dir}')
_run(f'ls -lh /kaggle/working/{zip_name}')
_run('df -h /kaggle/working')
# stop watcher (best-effort)
try:
    os.kill(int(open('/tmp/watcher.pid').read()), 9)
except Exception as e:
    print(f'watcher kill skipped: {e}')

In [ ]:
# Summary
import pandas as pd, os
if os.path.exists('/kaggle/working/logs/mem-trace.csv'):
    df = pd.read_csv('/kaggle/working/logs/mem-trace.csv')
    print(f'Samples: {len(df)}')
    if len(df):
        print('5 lowest MemAvail (MB):')
        print(df.nsmallest(5,'MemAvail_MB')[['datetime','MemAvail_MB','TopRSS_MB','TopProc']].to_string())
        print('5 highest TopRSS (MB):')
        print(df.nlargest(5,'TopRSS_MB')[['datetime','TopRSS_MB','TopProc']].to_string())
!ls -lh /kaggle/working/